In [1]:
import os
CODE_DIR = "/kaggle/working"
DATASET  = "MinhDuk/Chunking_571_Skynet_combined"
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_LEROBOT_HOME"] = "/tmp/lerobot"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"

In [2]:
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"
# lerobot needs av.option -> av 19 lacks it, pin av 15.x AFTER lerobot is installed
!pip install -q -U "av>=15,<16"
!python -c "import av, av.option; print('av', av.__version__, 'OK')"
!nvidia-smi -L

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
av 15.1.0 OK
GPU 0: Tesla T4 (UUID: GPU-9524f258-f2cd-e199-db50-36974898f0b8)
GPU 1: Tesla T4 (UUID: GPU-187c0b28-1207-55c6-7c9f-49a2307a684b)


In [3]:
%%writefile /kaggle/working/gate_patch.py
#!/usr/bin/env python
"""
Patch LeRobot's SmolVLA in place: Gated Attention (Qiu et al., 2025, Qwen) in the ACTION EXPERT.

  Y' = Y * sigmoid(X @ W + b)      (paper position "G1": after SDPA, before o_proj)

  * X = pre-normed hidden state of the query token (expert/action token) -> query-dependent gate
  * head-specific, elementwise by default (one score per head-dim channel; best variant in the paper)
  * applied in EVERY expert layer (self-attention layers and cross-attention layers)
  * VLM layers / prefix tokens are never gated
  * gate init: W ~ N(0, 0.02), b = +4  ->  sigmoid ~ 0.98 (near identity, so the pretrained
    smolvla_base expert is not destroyed at step 0, but gradients still flow)
  * SmolVLM + expert are run in fp32 (T4 has no native bf16)

Nothing else is changed (masks, RoPE, KV cache, flow-matching loss, optimizer, scheduler).

    python gate_patch.py            # patch installed lerobot
    python gate_patch.py --dir D    # patch a given smolvla folder
    python gate_patch.py --revert   # restore .orig backups
"""
import argparse
import importlib.util
import pathlib
import shutil
import sys


def find_dir() -> pathlib.Path:
    spec = importlib.util.find_spec("lerobot")
    if spec is None or not spec.submodule_search_locations:
        sys.exit("lerobot is not installed.")
    return pathlib.Path(list(spec.submodule_search_locations)[0]) / "policies" / "smolvla"


def sub(text: str, old: str, new: str, name: str) -> str:
    n = text.count(old)
    if n != 1:
        sys.exit(f"[patch failed] anchor '{name}' found {n}x (expected 1). "
                 f"Your lerobot version differs from the one this patch was written for.")
    return text.replace(old, new)


# ------------------------------------------------------------------ configuration_smolvla.py
def patch_config(t: str) -> str:
    return sub(
        t,
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n',
        '    compile_mode: str = "max-autotune"  # Torch compile mode\n'
        "\n"
        "    # Gated Attention (sigmoid gate after SDPA) in every layer of the action expert\n"
        "    use_gated_attn: bool = False\n"
        "    gated_attn_headwise: bool = False  # False: elementwise (n x heads x head_dim), True: one scalar per head\n"
        "    gated_attn_bias_init: float = 4.0  # sigmoid(4) = 0.982 -> near-identity start from the pretrained expert\n",
        "config.fields",
    )


# ------------------------------------------------------------------ smolvlm_with_expert.py
GATE_CLASS = '''class AttnOutputGate(nn.Module):
    """Gated Attention, position G1 (Qiu et al. 2025): Y' = Y * sigmoid(X W + b).
    Y = SDPA output [B, L, H*D] (before o_proj), X = normed hidden state of the query tokens [B, L, E].
    Head-specific. Elementwise: one score per channel (H*D). Headwise: one score per head (H)."""

    def __init__(self, hidden_size, num_heads, head_dim, headwise=False, bias_init=4.0):
        super().__init__()
        self.headwise = headwise
        self.head_dim = head_dim
        self.proj = nn.Linear(hidden_size, num_heads if headwise else num_heads * head_dim)
        nn.init.normal_(self.proj.weight, std=0.02)
        nn.init.constant_(self.proj.bias, bias_init)

    def forward(self, x, att_out):
        g = torch.sigmoid(self.proj(x.to(self.proj.weight.dtype)))
        if self.headwise:
            g = g.repeat_interleave(self.head_dim, dim=-1)
        return att_out * g.to(att_out.dtype)


class SmolVLMWithExpertModel(nn.Module):'''

GATE_BUILD = '''        # Remove unused embed_tokens
        self.lm_expert.embed_tokens = None

        # Gated attention: one gate per action-expert layer (self- and cross-attention layers)
        self.use_gated_attn = use_gated_attn
        self.attn_gates = nn.ModuleDict()
        if use_gated_attn:
            assert self.num_expert_layers == self.num_vlm_layers, (
                "Gated attention assumes num_expert_layers == num_vlm_layers (default config)"
            )
            for _i in range(self.num_expert_layers):
                self.attn_gates[str(_i)] = AttnOutputGate(
                    lm_expert_config.hidden_size,
                    lm_expert_config.num_attention_heads,
                    lm_expert_config.head_dim,
                    gated_attn_headwise,
                    gated_attn_bias_init,
                )
'''


def patch_vlm(t: str) -> str:
    t = sub(t, "class SmolVLMWithExpertModel(nn.Module):", GATE_CLASS, "class")
    t = sub(t, 'torch_dtype="bfloat16",', 'torch_dtype="float32",', "fp32 loader")
    t = sub(
        t,
        '        device: str = "auto",\n    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        '        device: str = "auto",\n'
        "        use_gated_attn: bool = False,\n"
        "        gated_attn_headwise: bool = False,\n"
        "        gated_attn_bias_init: float = 4.0,\n"
        '    ):\n        super().__init__()\n        require_package("transformers", extra="smolvla")',
        "init signature",
    )
    t = sub(
        t,
        "        # Remove unused embed_tokens\n        self.lm_expert.embed_tokens = None\n",
        GATE_BUILD,
        "gate build",
    )
    # --- self-attention layers (joint prefix+suffix attention): remember the normed expert hidden state
    t = sub(
        t,
        "        query_states = []\n        key_states = []\n        value_states = []\n",
        "        gate_x = None\n        query_states = []\n        key_states = []\n        value_states = []\n",
        "gate_x init",
    )
    t = sub(
        t,
        "            hidden_states = layer.input_layernorm(hidden_states)\n",
        "            hidden_states = layer.input_layernorm(hidden_states)\n"
        "            if i == 1 and self.use_gated_attn:  # expert (action) tokens only\n"
        "                gate_x = hidden_states\n",
        "gate_x capture",
    )
    t = sub(
        t,
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states\n"
        "        )\n"
        "        return [att_output], past_key_values\n",
        "        att_output = attention_interface(\n"
        "            attention_mask_, batch_size, head_dim, query_states, key_states, value_states\n"
        "        )\n"
        "        if gate_x is not None:  # gate only the expert tokens (the last Ls query positions)\n"
        "            _ls = gate_x.shape[1]\n"
        "            _gated = self.attn_gates[str(layer_idx)](gate_x, att_output[:, -_ls:])\n"
        "            att_output = torch.cat([att_output[:, :-_ls], _gated], dim=1)\n"
        "        return [att_output], past_key_values\n",
        "self-attn gate apply",
    )
    # --- cross-attention layers (expert queries attend to the VLM KV)
    t = sub(
        t,
        "                expert_value_states,\n            )\n            att_outputs.append(att_output)\n",
        "                expert_value_states,\n            )\n"
        "            if self.use_gated_attn:\n"
        "                att_output = self.attn_gates[str(layer_idx)](expert_hidden_states, att_output)\n"
        "            att_outputs.append(att_output)\n",
        "cross-attn gate apply",
    )
    return t


# ------------------------------------------------------------------ modeling_smolvla.py
def patch_modeling(t: str) -> str:
    t = sub(
        t,
        '            device=self.config.device if self.config.device is not None else "auto",\n        )\n',
        '            device=self.config.device if self.config.device is not None else "auto",\n'
        "            use_gated_attn=self.config.use_gated_attn,\n"
        "            gated_attn_headwise=self.config.gated_attn_headwise,\n"
        "            gated_attn_bias_init=self.config.gated_attn_bias_init,\n"
        "        )\n",
        "pass flags",
    )
    t = sub(
        t,
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n",
        "        self.prefix_length = self.config.prefix_length\n        self.rtc_processor = rtc_processor\n"
        "        self.float()  # fp32 everywhere (T4 has no native bf16)\n",
        "fp32 cast",
    )
    return t


PATCHES = {
    "configuration_smolvla.py": patch_config,
    "smolvlm_with_expert.py": patch_vlm,
    "modeling_smolvla.py": patch_modeling,
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", default=None)
    ap.add_argument("--revert", action="store_true")
    args = ap.parse_args()
    d = pathlib.Path(args.dir) if args.dir else find_dir()
    print("SmolVLA dir:", d)

    if args.revert:
        for name in PATCHES:
            bak = d / (name + ".orig")
            if bak.exists():
                shutil.copy(bak, d / name)
                print("restored", name)
        return

    if "use_gated_attn" in (d / "configuration_smolvla.py").read_text():
        print("Already patched, nothing to do.")
        return

    new = {}
    for name, fn in PATCHES.items():  # patch in memory first, write only if ALL succeed
        new[name] = fn((d / name).read_text())
    for name, text in new.items():
        shutil.copy(d / name, d / (name + ".orig"))
        (d / name).write_text(text)
        print("patched", name)


if __name__ == "__main__":
    main()

Writing /kaggle/working/gate_patch.py


In [4]:
%%writefile /kaggle/working/gate_sanity_test.py
#!/usr/bin/env python
"""
Safety tests for the gated-attention patch (run AFTER gate_patch.py, BEFORE training).

    python gate_sanity_test.py            # CPU unit tests of the gate module
    python gate_sanity_test.py --model    # + model-level tests on lerobot/smolvla_base (needs a GPU)

Model-level:
  (1) gate forced to 1.0 (W=0, b=50) reproduces the baseline exactly (train loss + sampled actions)
  (2) random gates: full uncached forward == prefill + cached denoise_step  (gate only touches expert tokens)
  (3) every gate (self- and cross-attn layers) gets a finite, non-zero gradient at init
"""
import argparse

import torch

from lerobot.policies.smolvla.smolvlm_with_expert import AttnOutputGate

MODEL_ID = "lerobot/smolvla_base"


def test_math():
    torch.manual_seed(0)
    B, L, E, H, D = 2, 7, 24, 3, 8
    x, y = torch.randn(B, L, E), torch.randn(B, L, H * D)
    for headwise in (False, True):
        g = AttnOutputGate(E, H, D, headwise=headwise, bias_init=0.0)
        torch.nn.init.normal_(g.proj.weight, std=0.5)
        out = g(x, y)
        s = torch.sigmoid(g.proj(x))
        if headwise:
            s = s.repeat_interleave(D, dim=-1)
        assert out.shape == y.shape and torch.allclose(out, y * s, atol=1e-6)
        # query dependence + causality of the gate: token t's output only depends on token t's x
        x2 = x.clone()
        x2[:, 4:] += 1.0
        assert torch.allclose(g(x, y)[:, :4], g(x2, y)[:, :4])
    g = AttnOutputGate(E, H, D, bias_init=4.0)
    assert (g(x, y) / y).mean() > 0.95, "default init should be near identity"
    print("[ok] math tests: elementwise/headwise formula, per-token dependence, near-identity init")


def fake_batch(policy, B=2, device="cuda"):
    from lerobot.utils.constants import ACTION, OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS, OBS_STATE

    cfg = policy.config
    batch = {}
    for key, ft in cfg.image_features.items():
        c, h, w = ft.shape
        batch[key] = torch.rand(B, c, h, w, device=device)
    batch[OBS_STATE] = torch.randn(B, cfg.robot_state_feature.shape[0], device=device)
    batch[ACTION] = torch.randn(B, cfg.chunk_size, cfg.action_feature.shape[0], device=device)
    batch[OBS_LANGUAGE_TOKENS] = torch.randint(0, 1000, (B, 16), device=device)
    batch[OBS_LANGUAGE_ATTENTION_MASK] = torch.ones(B, 16, dtype=torch.bool, device=device)
    return batch


def load_policy(use_gate, device="cuda"):
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.smolvla import SmolVLAPolicy

    cfg = PreTrainedConfig.from_pretrained(MODEL_ID)
    cfg.device = device
    cfg.use_gated_attn = use_gate
    return SmolVLAPolicy.from_pretrained(MODEL_ID, config=cfg).to(device)


def unpack(policy, batch):
    from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

    images, img_masks = policy.prepare_images(batch)
    state = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)
    return images, img_masks, batch[OBS_LANGUAGE_TOKENS], batch[OBS_LANGUAGE_ATTENTION_MASK], state, actions


@torch.no_grad()
def uncached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    se, spm, sam = vf.embed_suffix(x_t, time)
    pad = torch.cat([ppm, spm], 1)
    att = torch.cat([pam, sam], 1)
    (_, suf), _ = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(pad, att),
        position_ids=torch.cumsum(pad, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, se],
        use_cache=False,
    )
    return vf.action_out_proj(suf[:, -vf.config.chunk_size :].float())


@torch.no_grad()
def cached_vt(vf, images, img_masks, lang, lm, state, x_t, time):
    from lerobot.policies.common.vla_utils import make_att_2d_masks

    pe, ppm, pam = vf.embed_prefix(images, img_masks, lang, lm, state=state)
    _, pkv = vf.vlm_with_expert.forward(
        attention_mask=make_att_2d_masks(ppm, pam),
        position_ids=torch.cumsum(ppm, 1) - 1,
        past_key_values=None,
        inputs_embeds=[pe, None],
        use_cache=True,
    )
    return vf.denoise_step(prefix_pad_masks=ppm, past_key_values=pkv, x_t=x_t, timestep=time)


def test_model():
    from lerobot.policies.common.flow_matching import FlowConvention, make_flow_matching_inputs

    dev = "cuda"
    base = load_policy(False, dev).eval()
    gated = load_policy(True, dev).eval()
    miss = [k for k in gated.state_dict() if k not in base.state_dict()]
    assert miss and all("attn_gates" in k for k in miss), f"unexpected extra keys: {miss[:5]}"
    gates = gated.model.vlm_with_expert.attn_gates
    print(f"[info] {len(miss)} new tensors, all attn_gates.*  ({len(gates)} gates, "
          f"{sum(p.numel() for p in gates.parameters()) / 1e6:.2f}M params)")
    assert next(gated.parameters()).dtype == torch.float32, "model is not fp32"

    torch.manual_seed(0)
    batch = fake_batch(base, device=dev)
    B = batch[next(iter(base.config.image_features))].shape[0]
    noise = torch.randn(B, base.config.chunk_size, base.config.max_action_dim, device=dev)
    time = torch.rand(B, device=dev) * 0.9 + 0.05
    ia, ib = unpack(base, dict(batch)), unpack(gated, dict(batch))

    # (1) gate == 1.0  ->  baseline, exactly
    for g in gates.values():
        g.proj.weight.data.zero_()
        g.proj.bias.data.fill_(50.0)
    with torch.no_grad():
        la = base.model.forward(*ia, noise, time)
        lb = gated.model.forward(*ib, noise, time)
        assert torch.allclose(la, lb, atol=1e-6), (la - lb).abs().max()
        sa = base.model.sample_actions(*ia[:5], noise=noise)
        sb = gated.model.sample_actions(*ib[:5], noise=noise)
        assert torch.allclose(sa, sb, atol=1e-6), (sa - sb).abs().max()
    print("[ok] (1) gate=1 reproduces the baseline (loss + sampled actions)")
    v_ref = None

    # (3) default init: every gate gets a finite non-zero gradient
    for g in gates.values():
        torch.nn.init.normal_(g.proj.weight, std=0.02)
        torch.nn.init.constant_(g.proj.bias, 4.0)
    gated.train()
    gated.zero_grad()
    loss = gated.model.forward(*ib, noise, time).mean()
    loss.backward()
    for k, g in gates.items():
        gr = g.proj.weight.grad
        assert gr is not None and torch.isfinite(gr).all() and gr.abs().sum() > 0, f"gate {k}: bad grad"
    assert torch.isfinite(loss)
    print(f"[ok] (3) all {len(gates)} gates (self+cross layers) get finite non-zero grads, loss={loss.item():.4f}")
    gated.eval()
    gated.zero_grad(set_to_none=True)

    # (2) random gates: uncached training path == prefill + cached denoise path
    for g in gates.values():
        torch.nn.init.normal_(g.proj.weight, std=0.5)
        torch.nn.init.normal_(g.proj.bias, std=1.0)
    x_t, _, _ = make_flow_matching_inputs(ib[5], noise, time, convention=FlowConvention.NOISE_AT_ONE)
    v_unc = uncached_vt(gated.model, *ib[:5], x_t, time)
    v_cac = cached_vt(gated.model, *ib[:5], x_t, time)
    err = (v_unc - v_cac).abs().max().item()
    assert err < 1e-3, f"train path and cached path disagree (max abs err {err:.3e})"
    for g in gates.values():
        g.proj.weight.data.zero_()
        g.proj.bias.data.fill_(50.0)
    v1 = uncached_vt(gated.model, *ib[:5], x_t, time)
    assert (v1 - v_unc).abs().max() > 1e-4, "gates have no effect"
    print(f"[ok] (2) uncached vs cached path agree (max abs err {err:.2e}); gates change the output")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", action="store_true")
    args = ap.parse_args()
    test_math()
    if args.model:
        test_model()
    print("ALL TESTS PASSED")

Writing /kaggle/working/gate_sanity_test.py


In [9]:
%%writefile /kaggle/working/gate_profile.py
#!/usr/bin/env python
"""
Parameter count + FLOPs for SmolVLA, baseline vs gated attention (run AFTER gate_patch.py).

    python gate_profile.py              # baseline vs gated, batch size 1
    python gate_profile.py --bs 8       # per-batch numbers for another batch size

FLOPs are measured with torch.utils.flop_counter.FlopCounterMode (counts matmul / bmm / conv / sdpa,
1 MAC = 2 FLOPs; elementwise ops like sigmoid/softmax are not counted).
  * train step = forward + backward of the flow-matching loss (what is actually executed, frozen VLM included)
  * inference  = embed prefix + VLM prefill once + `num_steps` (10) expert denoise steps = one action chunk
"""
import argparse
import gc
from collections import OrderedDict

import torch
from torch.utils.flop_counter import FlopCounterMode

from gate_sanity_test import fake_batch, load_policy, unpack


def count_params(policy):
    groups = OrderedDict(
        [("VLM vision encoder", 0), ("VLM connector", 0), ("VLM text model (16 layers)", 0),
         ("Action expert (lm_expert)", 0), ("Attention gates (new)", 0), ("Projections (state/action/time)", 0)]
    )
    train = OrderedDict((k, 0) for k in groups)
    for n, p in policy.named_parameters():
        if "attn_gates" in n:
            k = "Attention gates (new)"
        elif "lm_expert" in n:
            k = "Action expert (lm_expert)"
        elif "vision_model" in n:
            k = "VLM vision encoder"
        elif "connector" in n:
            k = "VLM connector"
        elif ".vlm." in n:
            k = "VLM text model (16 layers)"
        else:
            k = "Projections (state/action/time)"
        groups[k] += p.numel()
        if p.requires_grad:
            train[k] += p.numel()
    return groups, train


def count_flops(policy, bs, dev):
    batch = fake_batch(policy, B=bs, device=dev)
    inp = unpack(policy, batch)

    policy.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        policy.model.sample_actions(*inp[:5])
    infer = fc.get_total_flops()

    policy.train()
    policy.zero_grad(set_to_none=True)
    with FlopCounterMode(display=False) as fc:
        policy.model.forward(*inp).mean().backward()
    train = fc.get_total_flops()
    policy.zero_grad(set_to_none=True)

    policy.eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        policy.model.forward(*inp)
    fwd = fc.get_total_flops()
    return {"forward (loss only)": fwd, "train step (fwd+bwd)": train, "inference (1 chunk, 10 steps)": infer}


def profile(use_gate, bs, dev):
    pol = load_policy(use_gate, dev)
    groups, trainable = count_params(pol)
    flops = count_flops(pol, bs, dev)
    del pol
    gc.collect()
    torch.cuda.empty_cache()
    return groups, trainable, flops


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bs", type=int, default=1)
    args = ap.parse_args()
    dev = "cuda"

    res = {name: profile(flag, args.bs, dev) for name, flag in [("baseline", False), ("gated", True)]}

    print(f"\n================ PARAMETERS ================")
    print(f"{'group':36s} {'baseline':>14s} {'gated':>14s}   (trainable gated)")
    gb, tb, _ = res["baseline"]
    gg, tg, _ = res["gated"]
    for k in gb:
        print(f"{k:36s} {gb[k]:>14,d} {gg[k]:>14,d}   {tg[k]:>14,d}")
    print(f"{'TOTAL':36s} {sum(gb.values()):>14,d} {sum(gg.values()):>14,d}   {sum(tg.values()):>14,d}")
    print(f"{'TRAINABLE':36s} {sum(tb.values()):>14,d} {sum(tg.values()):>14,d}")
    extra = sum(gg.values()) - sum(gb.values())
    print(f"-> gates add {extra:,d} params ({100 * extra / sum(gb.values()):.2f}% of the baseline total, "
          f"{100 * extra / sum(tb.values()):.2f}% of its trainable params)")

    print(f"\n================ FLOPs (batch size {args.bs}) ================")
    fb, fg = res["baseline"][2], res["gated"][2]
    print(f"{'':32s} {'baseline GFLOPs':>16s} {'gated GFLOPs':>14s} {'overhead':>10s}")
    for k in fb:
        print(f"{k:32s} {fb[k] / 1e9:>16,.2f} {fg[k] / 1e9:>14,.2f} {100 * (fg[k] / fb[k] - 1):>9.2f}%")
    print("(1 MAC = 2 FLOPs; matmul/bmm/conv/sdpa only)")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/gate_profile.py


In [10]:
!python {CODE_DIR}/gate_patch.py

SmolVLA dir: /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla
Already patched, nothing to do.


In [11]:
# safety tests: gate math (CPU) + model-level (gate=1 == baseline, cached == uncached path, grads on all gates)
!python {CODE_DIR}/gate_sanity_test.py --model

[ok] math tests: elementwise/headwise formula, per-token dependence, near-identity init
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2565.68it/s]
Reducing the number of VLM layers to 16 ...
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2583.68it/s]
Reducing the number of VLM layers to 16 ...
[info] 32 new tensors, all attn_gates.*  (16 gates, 11.07M params)
[ok] (1) gate=1 reproduces the baseline (loss + sampled actions)
[ok] (3) all 16 gates (self+cross layers) get finite non-zero grads, loss=0.9125
[ok] (2) uncached vs cached path agree (max abs err 1.91e-06); gates change the output
ALL TESTS PASSED


In [12]:
!python {CODE_DIR}/gate_profile.py --bs 1

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2602.30it/s]
Reducing the number of VLM layers to 16 ...
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Loading weights: 100%|██████████████████████| 489/489 [00:00<00:00, 2554.87it/s]
Reducing the number of VLM layers to 16 ...

================ PARAMETERS ================
group                                      baseline          gated   (trainable gated)
VLM vision encoder                       86,433,024     86,433,024                0
VLM connector                            11,796,480     11,796,480                0
VLM text model (16 layers)              251,935,680    251,935,680                0
Action expert (lm_expert)                98,245,840     98,245,840       98,245,840
Attention gates (new)                             0     11,074,560       11,074,560
Proj

In [13]:
import os, shutil, subprocess, sys

TRAIN  = shutil.which("lerobot-train")
STEPS  = 20000
OUT    = "/kaggle/working/outputs/smolvla_gated_attn"
RENAME = ('{"observation.images.cam_head":"observation.images.camera1",'
          '"observation.images.cam_wrist":"observation.images.camera2",'
          '"observation.images.cam_side":"observation.images.camera3"}')

cmd = [
    "accelerate", "launch", "--multi_gpu", "--num_processes=2", "--mixed_precision=no", TRAIN,
    "--policy.path=lerobot/smolvla_base",
    "--policy.use_gated_attn=true",
    "--policy.gated_attn_headwise=false",   # false = elementwise gate (best in the paper); true = one scalar per head
    "--policy.gated_attn_bias_init=4.0",    # sigmoid(4)=0.98 -> starts ~identity from the pretrained expert
    f"--policy.scheduler_decay_steps={STEPS}",
    "--policy.push_to_hub=false",
    f"--dataset.repo_id={DATASET}",
    "--dataset.revision=main",
    "--dataset.video_backend=pyav",
    f"--rename_map={RENAME}",
    "--batch_size=8",
    "--num_workers=4",
    f"--steps={STEPS}",
    "--save_freq=2500",
    "--log_freq=50",
    f"--output_dir={OUT}",
    "--job_name=smolvla_gated_attn",
    "--seed=1000",
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
sys.exit(proc.wait())

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
INFO 2026-11-08 10:36:05 ot_train.py:565 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'mixed_precision_policy': {'cast_forward_i

KeyboardInterrupt: 